In [0]:
%sql
-- deduplicating rows -> since im filtering only to episode 1, there are never going to be duplicates there. so filtering takes care of the deduplicating issue! 
-- filtering to only episode 1
-- filtering precision category (estimated LMP precision indicator)
-- filtering out people who appear initially in their pregnancy and never return. 
-- recorded_episode_length >= X  (eg., 1 month) AND recorded_episode_end >= inferred_episode_start + Y days (e.g., 90 days, so at least through the first trimester) AND n_visits >= Y (e.g., 2 visits with unique visit_start_date)

-- made with help from claude
-- SELECT COUNT(*)
CREATE OR REPLACE TABLE catalog.schema.step7_filtered_cohort AS
SELECT pe.*
FROM catalog.schema.step7_pregnancy_episodes_with_metadata pe
JOIN (
    SELECT vo.person_id, COUNT(DISTINCT vo.visit_start_date) AS n_visits
    FROM catalog.omop.visit_occurrence vo
    JOIN catalog.schema.step7_pregnancy_episodes_with_metadata pe2
        ON vo.person_id = pe2.person_id
        AND vo.visit_start_date BETWEEN LEAST(pe2.inferred_episode_start, pe2.recorded_episode_start) AND GREATEST(pe2.inferred_episode_end, pe2.recorded_episode_end)
    GROUP BY vo.person_id
) vo ON pe.person_id = vo.person_id AND vo.n_visits >= 2
JOIN (
    SELECT 
        vo.person_id,
        SUM(CASE WHEN vo.visit_start_date >= LEAST(pe2.inferred_episode_start, pe2.recorded_episode_start) + INTERVAL 14 weeks
                   AND vo.visit_start_date <  LEAST(pe2.inferred_episode_start, pe2.recorded_episode_start) + INTERVAL 28 weeks
                  THEN 1 ELSE 0 END) AS n_visits_t2,
        SUM(CASE WHEN vo.visit_start_date >= LEAST(pe2.inferred_episode_start, pe2.recorded_episode_start) + INTERVAL 28 weeks
                   AND vo.visit_start_date <  LEAST(pe2.inferred_episode_start, pe2.recorded_episode_start) + INTERVAL 42 weeks
                  THEN 1 ELSE 0 END) AS n_visits_t3
    FROM (
        SELECT DISTINCT person_id, visit_start_date FROM catalog.omop.visit_occurrence
    ) vo
    JOIN catalog.schema.step7_pregnancy_episodes_with_metadata pe2
        ON vo.person_id = pe2.person_id
    GROUP BY vo.person_id
) tri ON pe.person_id = tri.person_id AND tri.n_visits_t2 >= 3 AND tri.n_visits_t3 >= 3
WHERE pe.episode_number = 1
AND pe.final_outcome_category != 'DELIV' -- null also excluded here
AND pe.precision_category NOT IN ('non-specific', 'week_poor-support')
AND GREATEST(pe.recorded_episode_end, pe.inferred_episode_end) >= LEAST(pe.inferred_episode_start, pe.recorded_episode_start) + INTERVAL '90 days'
AND pe.recorded_episode_length >= 1

In [0]:
# %sql
# SELECT p.specialty_concept_id, c.concept_name, ca.care_site_name, COUNT(DISTINCT vo.person_id) AS n_patients
# FROM catalog.schema.step7_filtered_cohort_with_apos pe
# JOIN catalog.omop.visit_occurrence vo
#     ON pe.person_id = vo.person_id
# JOIN catalog.omop.provider p
#     ON p.provider_id = vo.provider_id
# JOIN catalog.omop.care_site ca
#     ON ca.care_site_id = vo.care_site_id
# JOIN catalog.omop.concept c
#     ON c.concept_id = p.specialty_concept_id
# WHERE vo.visit_start_date >= DATE_SUB(pe.inferred_episode_start, 180)
#     AND vo.visit_start_date < pe.inferred_episode_start
# GROUP BY p.specialty_concept_id, c.concept_name, ca.care_site_name
# ORDER BY n_patients DESC;

# -- SELECT COUNT(*)
# -- FROM catalog.schema.step7_filtered_cohort_with_apos pe
# -- LEFT JOIN catalog.omop.visit_occurrence vo
# --     ON pe.person_id = vo.person_id
# --     AND vo.visit_start_date < pe.inferred_episode_start
# -- WHERE pe.person_id IS NULL


In [0]:
%sql
SELECT 'accreta' AS match_reason, concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%accreta%'

UNION ALL

SELECT 'previa', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%previa%'

UNION ALL

SELECT 'hellp', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%hellp%'

UNION ALL

SELECT 'hemolysis_elevated_liver_low_platelet', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%hemolysis%elevated%liver%enzymes%low%platelet%'

UNION ALL

SELECT 'miscarriage', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%miscarriage%'

UNION ALL

SELECT 'stillbirth', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%stillbirth%'

UNION ALL

SELECT 'hyperemesis_gravidarum', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%hyperemesis gravidarum%'

UNION ALL

SELECT 'fetal_growth_abnormal', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%fetal growth%' AND concept_name NOT ILIKE '%normal%'

UNION ALL

SELECT 'uterine_rupture', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%uterine%rupture%'

UNION ALL

SELECT 'chronic_hypertension', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE '%chronic hypertension%'

UNION ALL

SELECT 'type_diabetes_mellitus', concept_id, concept_name, concept_code, vocabulary_id
FROM catalog.omop.concept
WHERE concept_name ILIKE 'type%diabetes%mellitus%'

ORDER BY match_reason, concept_id

In [0]:
%sql
-- FROM CLAUDE: phenotyping using filtered + partitioned tables
CREATE OR REPLACE TABLE catalog.schema.step7_filtered_cohort_with_apos AS

WITH 

control_additional_exclusion_codes AS (
    -- FIXED, explicit concept_id list (extracted once from the ILIKE-based 
    -- search, reviewed, and locked in below for determinism)
    SELECT explode(array(
        -- accreta
        1423840, 3068519, 3144409, 3280606, 3386454, 3401607, 3545476, 37091876, 37111273, 40366952, 40392164, 4140392, 4289303, 4344181, 45480496, 45520298, 45563147, 45582553, 45587362, 45601863, 45606628, 45615656, 45932027, 45953475, 506554,
        -- previa
        10159, 1423560, 1423561, 1423562, 1423565, 1423566, 1423567, 1424389, 1571824, 1571825, 1571826, 192372, 193264, 194710, 196751, 200149, 201359, 201410, 35210414, 35210549, 37091883, 37091886, 37108740, 37108741, 37111254, 37200990, 37200991, 37200992, 37200993, 37200994, 37200995, 37200996, 37200997, 37200998, 37200999, 40225421, 40318646, 40318650, 40322926, 40393455, 40393888, 40393889, 40393926, 40398827, 4047712, 4049740, 4058399, 4058401, 4060161, 4062412, 4062413, 4064846, 4070410, 4081720, 4099747, 4143074, 4172142, 4196670, 4217209, 4238647, 4260163, 4269793, 4284028, 4338977, 435021, 435609, 442171, 44784542, 44784543, 44784544, 44784545, 44821058, 44823282, 44824386, 44825587, 44826772, 44826773, 44826820, 44829098, 44829099, 44829151, 44831372, 44832519, 44833712, 44833898, 45430620, 45534299, 45548803, 45548823, 45553567, 45563170, 45563171, 45572886, 45577646, 45597008, 45601865, 45601866, 45601867, 45601868, 45601887, 45601888, 45615657, 45618436, 45908739, 45908740, 45909717, 45914859, 45914861, 45925517, 45932025, 45932335, 45939417, 45939505, 45943446, 45943937, 45946424, 45948528, 45950966, 45953473,
        -- hellp
        1423100, 19150302, 37091704, 37200971, 37200972, 37616219, 42495178, 42619326, 4316372, 4345513, 45426980, 45563053, 45572763, 45577561, 45582448, 45618287, 45880616, 45926665, 506552,
        -- hemolysis/elevated liver/low platelet
        37111517, 40394357,
        -- miscarriage
        1006897, 1006899, 1340198, 1340199, 1340200, 1340201, 1340202, 1340203, 1422828, 1423043, 192678, 193820, 194704, 195317, 195594, 196746, 3071939, 3073861, 3091185, 3106374, 3106402, 3109556, 3126522, 3146699, 3146700, 3146701, 3146702, 3146703, 3146705, 3146706, 3146707, 3146708, 3146709, 3155673, 3156411, 3177643, 3190176, 3203261, 3207766, 3224243, 3229853, 3230735, 3234137, 3245203, 3247772, 3251725, 3261924, 3265606, 3272338, 3275611, 3276726, 3282199, 3283997, 3296004, 3296321, 3299330, 3301303, 3305004, 3306621, 3314392, 3314407, 3315349, 3317253, 3327545, 3334035, 3344304, 3348150, 3352768, 3357857, 3373283, 3377414, 3378424, 3380831, 3382100, 3383327, 3388016, 3390804, 3394159, 3399524, 3410357, 3415111, 3415690, 3417152, 3450900, 3463871, 3533659, 3544968, 3559027, 3559028, 35810317, 35810403, 35826466, 3956914, 3956915, 3957743, 3957744, 3957745, 3958835, 4001856, 4003276, 4006322, 4006806, 4011615, 40303215, 4031837, 40318617, 40318618, 40318645, 40318677, 4032055, 4033096, 40392527, 4041728, 4047775, 4049891, 4051244, 40563037, 40566187, 4058553, 4058873, 4059481, 4059482, 4060188, 4060189, 4060190, 4060191, 4060192, 4062565, 4064536, 4067106, 4074867, 40767414, 40767416, 4078393, 4085627, 4087102, 4087135, 4090055, 4095945, 4095946, 4130670, 4146772, 4146773, 4146774, 4146775, 4149423, 4167685, 4168445, 4170121, 4170457, 4175137, 4178173, 4179796, 4180568, 4189350, 4189550, 4194113, 4194423, 4195406, 4200070, 4202200, 4206899, 4209126, 4211863, 4216342, 4217670, 4219469, 4221853, 4224597, 4224646, 4224903, 4238207, 4244261, 4244408, 4247270, 4252252, 42537157, 42537191, 42537759, 42537769, 42537952, 4253804, 4261327, 4266998, 4269081, 4274315, 4278107, 4285746, 42872817, 4290406, 4294259, 4302162, 4309780, 4310654, 4311698, 4313166, 4321536, 4323612, 432677, 432682, 432689, 4339107, 434094, 434413, 434689, 435000, 43530904, 43530979, 435318, 436176, 436742, 436748, 437045, 437618, 437920, 437933, 438211, 438805, 439085, 439688, 440457, 440460, 441348, 441629, 441630, 441631, 441632, 441910, 441921, 44783770, 44807496, 44807497, 44809051, 44809902, 45420542, 45430312, 45433621, 45433622, 45434989, 45434990, 45434991, 45443519, 45443521, 45443522, 45446857, 45456932, 45458244, 45462174, 45463679, 45465057, 45468034, 45468464, 45471794, 45473713, 45473714, 45473716, 45483714, 45487101, 45493641, 45493642, 45501622, 45503617, 45510221, 45511197, 45511607, 45513607, 45513608, 45516999, 45517001, 45523543, 45530138, 45757198, 45876808, 45879584, 45912009, 45917387, 45928918, 45933721, 45937965, 45941862, 46269746, 46269812, 46269813, 46274066, 601838, 609181, 609182, 764546, 76482,
        -- stillbirth
        1006897, 1006899, 1012637, 1434553, 1434607, 1434608, 1434618, 1434619, 3055920, 3056132, 3058166, 3069766, 3078849, 3082386, 3091184, 3098594, 3098807, 3098809, 3112741, 3130794, 3138637, 3163329, 3171410, 3180846, 3225106, 3230060, 3239302, 3251223, 3266215, 3288521, 3309814, 3315123, 3317166, 3318935, 3360601, 3366500, 3375427, 3387970, 3394394, 3419182, 35210683, 35225109, 3529111, 3546886, 3546887, 3573308, 3580525, 35809774, 35809777, 35809800, 35810317, 35810402, 35827762, 35827763, 35827764, 36310132, 37017027, 37097584, 37614864, 38004558, 38004648, 4014454, 4028787, 40303214, 40310703, 40311787, 40311790, 40323449, 40333669, 4034159, 40357697, 40406337, 40406338, 40406339, 40406694, 40515177, 40515178, 40524863, 4059480, 4060241, 40629514, 40767414, 40767416, 4092290, 4092786, 4103876, 4217655, 42505413, 42505414, 42505415, 42505416, 42505417, 42598765, 4261819, 4287409, 4300979, 4301269, 4321563, 4345683, 443213, 443463, 44792586, 44795842, 44798047, 44800562, 44817096, 45427300, 45432083, 45453860, 45468961, 45474897, 45480790, 45504739, 45505373, 45508243, 45510534, 45517957, 45529278, 45571347, 45612361, 45770031, 45773593, 45773710, 45773824, 45882030, 45893353, 45916063, 45922368, 45927021, 45934380, 45955906, 764548,
        -- hyperemesis gravidarum
        1423126, 1423127, 1423128, 1423129, 1423130, 3106390, 3106391, 3126761, 3126763, 3126764, 3126765, 3137325, 3143667, 3168505, 3172796, 3265911, 3277758, 3312084, 3332911, 3393691, 35209555, 35209556, 3536332, 3573293, 3573294, 37091716, 37091717, 37606448, 37613858, 4030429, 4030872, 40318663, 40318664, 40318665, 4032345, 40355721, 40358546, 40394367, 40422474, 4058089, 4058090, 4058091, 4098582, 4142340, 4227141, 42495191, 42495192, 42538165, 4342137, 436166, 436173, 436485, 440788, 44824396, 44826777, 44827936, 44827937, 44830213, 44831377, 44833716, 44837236, 45426982, 45433637, 45470486, 45493654, 45496917, 45500266, 45500267, 45500268, 45543916, 45596924, 45617922, 45912015, 45915679, 45915680, 45920838, 45920875, 45936935, 45939573, 45945922,
        -- fetal growth abnormal
        10084, 10085, 10086, 10087, 10088, 10089, 10090, 10091, 1326879, 1423486, 1423487, 1423490, 1423492, 1423493, 1424463, 1424474, 1424475, 1434590, 1434591, 1434592, 1435939, 1571788, 1571790, 1571791, 1571972, 3059940, 3067018, 3070333, 3075500, 3127108, 3130572, 3130573, 3143012, 3143013, 3171222, 3270942, 3520078, 3533905, 3564687, 36713463, 36716982, 37079460, 37091849, 37091850, 37092420, 37092424, 37097578, 37116878, 37604082, 37604083, 37607566, 37611016, 37612731, 4024562, 40357225, 40397730, 40404551, 40404555, 40406712, 40475170, 4047728, 4047729, 40521443, 40525250, 40535132, 4066354, 4070521, 4082628, 4082672, 4087113, 4124318, 4137949, 4146846, 42495349, 42495350, 42495704, 42495708, 42505403, 42784276, 42791281, 42872406, 43021775, 4328385, 4334299, 4334902, 4345289, 44794721, 44795944, 44796882, 44820410, 44821223, 44822275, 44822276, 44823267, 44823427, 44823916, 44825610, 44825780, 44826804, 44826805, 44828131, 44828132, 44829275, 44829278, 44831411, 44833751, 44834912, 44836104, 44836270, 44836271, 44836272, 45447179, 45470511, 45470821, 45470822, 45474899, 45477395, 45477397, 45484936, 45500587, 45506934, 45534248, 45539161, 45539162, 45542400, 45543979, 45543980, 45543981, 45543982, 45543983, 45543984, 45543985, 45548774, 45548775, 45552077, 45553530, 45553531, 45553532, 45558276, 45558277, 45558278, 45563128, 45563129, 45567940, 45567941, 45567942, 45567943, 45567944, 45572824, 45572825, 45572826, 45572827, 45572828, 45577608, 45577609, 45577610, 45577611, 45577612, 45577613, 45577614, 45582515, 45582516, 45582517, 45582518, 45582519, 45582520, 45582608, 45587336, 45587337, 45587338, 45587339, 45587340, 45592238, 45592239, 45592240, 45596987, 45597064, 45601839, 45601840, 45601841, 45601842, 45606595, 45606596, 45614224, 45938514, 45949312, 45949406, 45954378, 45954530, 45956064, 506752, 608369, 72693, 80204, 81636,
        -- uterine rupture
        1422529, 1423805, 1423808, 3214158, 3244985, 35210550, 37110285, 37110289, 42108665, 42110656, 42112393, 42115458, 42117983, 42121775, 42122590, 42123341, 42124116, 42124770, 42125276, 42126695, 42132199, 42133023, 42136501, 42139556, 42142137, 42145962, 42146465, 42147266, 42148007, 42148673, 42149154, 42150902, 42154393, 42156192, 42158813, 42162174, 42164883, 42168663, 42169249, 42169979, 42170267, 42171596, 42172279, 42173392, 42177422, 42178447, 42188067, 42188941, 42191411, 42193694, 42194735, 42195146, 42196194, 42196636, 42198055, 42198519, 42199604, 42200183, 42201752, 42203149, 42203801, 42204588, 42206258, 42207678, 42209259, 42210105, 42210161, 42211934, 42214154, 42217941, 42219521, 42221947, 42224552, 42228944, 42230027, 42230583, 42231839, 42231981, 42232857, 42234217, 42238463, 42241701, 42243768, 42246395, 42248250, 42252436, 42253166, 42253985, 42254745, 42255503, 42256146, 42258614, 42269917, 42273919, 42276389, 42278280, 42281398, 42285312, 42285922, 42286880, 42287561, 42289070, 42289765, 42290674, 42295953, 42297605, 42300111, 42319031, 42320143, 42320602, 42321406, 42322645, 42323097, 42323853, 42324493, 42325276, 42330573, 42332533, 42360205, 42392325, 42392360, 42392455, 42392502, 42392537, 4344766, 45612147, 45880870,
        -- chronic hypertension
        1423082, 1423083, 3227775, 3287690, 3337784, 37091695, 37614524, 4080325, 4227607, 42495169, 4291933, 4304837, 45572762, 45910319, 45920724, 45925579, 45950160,
        -- type X diabetes mellitus
        1326491, 1326492, 1326493, 1409105, 1409106, 1409107, 1409111, 1409112, 1409116, 1409117, 1409119, 1409120, 1409124, 1409125, 1409131, 1409132, 1409138, 1409139, 1409143, 1409144, 1409145, 1409146, 1409147, 1409148, 1409150, 1409151, 1409152, 1409156, 1409157, 1409161, 1409162, 1409164, 1409165, 1409169, 1409170, 1409177, 1409178, 1409184, 1409185, 1409189, 1409190, 1409191, 1409192, 1409193, 1409194, 1567940, 1567941, 1567942, 1567943, 1567944, 1567945, 1567946, 1567947, 1567948, 1567949, 1567950, 1567951, 1567952, 1567953, 1567954, 1567955, 1567956, 1567957, 1567958, 1567959, 1567960, 1567961, 1567962, 1567963, 1567964, 1567965, 1567966, 1567967, 1567968, 1567969, 1567970, 1567971, 192278, 198995, 201254, 201826, 3059915, 3062976, 3064697, 3073041, 3083403, 3084184, 3121789, 3121798, 3121799, 3121800, 3121801, 3121802, 3121803, 3121806, 3121815, 3121816, 3121817, 3121818, 3121820, 3121821, 3121822, 3147259, 3151599, 3151600, 3151601, 3157460, 3157461, 3157462, 3157960, 3157995, 3197357, 3250509, 3285235, 3299249, 3301650, 3356296, 3360835, 3363857, 3386734, 3392131, 3409032, 3438725, 3450024, 3464411, 3472292, 35206877, 35206878, 35206879, 35206880, 35206881, 35206882, 3522463, 3522464, 3522465, 3522467, 3522468, 3522470, 3522471, 3522472, 3522474, 3522475, 3522476, 3522477, 3522478, 3522479, 3522630, 3522660, 3522661, 3522662, 3522803, 3522920, 3522921, 3522922, 3522923, 3522924, 3522925, 3522926, 3522928, 3522929, 3522930, 3522931, 3522932, 3522934, 3522935, 3522936, 3522937, 3522938, 3524295, 3550872, 3550980, 3550981, 3551089, 3551090, 3551092, 3551093, 3551094, 3569411, 3570232, 3571423, 3572830, 3572831, 3572832, 3572833, 3572834, 3573862, 3575709, 3575710, 3575711, 3575712, 3575713, 3575714, 3575715, 3575716, 3575717, 3575718, 3575719, 3575720, 3575721, 3575722, 3575723, 3575724, 3575725, 3575726, 3576222, 3576223, 3576224, 3576225, 3576226, 3576228, 3576229, 3576230, 3576231, 3576232, 3576233, 3576234, 3576235, 3576237, 3576238, 3576239, 3576241, 3576260, 3576524, 3577597, 3586459, 3586461, 36674344, 37081998, 37081999, 37082002, 37082005, 37082008, 37082011, 37082014, 37082017, 37082020, 37082028, 37082031, 37082034, 37082035, 37082039, 37082042, 37082045, 37082048, 37082051, 37082055, 37082058, 37082065, 37082068, 37200141, 37200142, 37200143, 37200144, 37200145, 37200146, 37200147, 37200148, 37200149, 37200150, 37200151, 37200152, 37200153, 37200154, 37200155, 37200156, 37200157, 37200158, 37200159, 37200160, 37200161, 37200162, 37200163, 37200164, 37200165, 37200166, 37200167, 37200168, 37200169, 37200170, 37200171, 37200172, 37200173, 37200174, 37200175, 37200176, 37200177, 37200178, 37200179, 37200180, 37200181, 37200182, 37200183, 37200184, 37200185, 37200186, 37200187, 37200188, 37200189, 37200190, 37200191, 37200192, 37200193, 37200194, 37200195, 37200196, 37200197, 37200198, 37200199, 37200200, 37200201, 37200202, 37200203, 37200204, 37200205, 37200206, 37200207, 37200208, 37200209, 37200210, 37200211, 37200212, 37200213, 37200214, 37200215, 37200216, 37200217, 37200218, 37200219, 37200220, 37200221, 37200222, 37200223, 37200224, 37200225, 37200226, 37200227, 37200228, 37200229, 37200230, 37200231, 37200232, 37200233, 37200234, 37200235, 37200236, 37200237, 37200238, 37200239, 37200240, 37200241, 37200242, 37200243, 37200244, 37200245, 37200246, 37200247, 37200248, 37200249, 37200250, 37200251, 37200252, 37200253, 37200254, 37605935, 37606171, 37606915, 37606916, 37606917, 37608133, 37608134, 37610276, 37610277, 37610278, 37610371, 37610971, 37611092, 37612619, 37612885, 37613482, 37614124, 37614125, 37614126, 37614312, 37614313, 37615585, 40386754, 40386755, 40386756, 40386757, 40386758, 40386762, 40386766, 40386767, 40386768, 40386769, 40386770, 40386771, 40386772, 40386773, 40386775, 40386778, 40386779, 40386780, 40386781, 40386786, 40386790, 40386791, 40386792, 40386793, 40386794, 40386795, 40386798, 40386799, 40386800, 40482801, 40484648, 40484649, 40485020, 40518515, 40518516, 40518517, 40518519, 40518525, 40518526, 40520430, 40520431, 40520436, 40543500, 40543501, 40543502, 40575609, 40575612, 40575613, 40584359, 40584911, 40585980, 4096668, 4099214, 4099215, 4099217, 4099651, 4151281, 4151282, 4152858, 4193704, 4198296, 4200873, 4200875, 4230254, 42485945, 42485946, 42485947, 42485948, 42485949, 42485950, 42485951, 42485952, 42485954, 42485955, 42485956, 42485957, 42485958, 42485959, 42485960, 42485961, 42485962, 42485963, 42485964, 42485965, 42485966, 42485967, 42485968, 42485969, 42485970, 42485971, 42485972, 42485973, 42485974, 42485975, 42485976, 42485977, 42485978, 42485979, 42485980, 42485981, 42485982, 42485984, 42485985, 42485986, 42485987, 42485988, 42485989, 42485990, 42485991, 42485992, 42485993, 42485994, 42485995, 42485996, 42485997, 42485998, 42485999, 42486000, 42486001, 42486002, 42486003, 42486004, 42486005, 42486006, 42486007, 42486008, 42486009, 42486010, 42486011, 42486012, 42486013, 42486014, 42486015, 42486016, 42486017, 42486018, 42486019, 42486020, 42486021, 42486022, 42486023, 42486024, 42486025, 42486026, 42486027, 42486028, 42486030, 42486031, 42486032, 42486033, 42486034, 42486035, 42486036, 42486037, 42689564, 42689565, 42689566, 42689567, 42689568, 42689569, 42690662, 42690670, 4295011, 4304377, 443412, 44810563, 44810565, 45420112, 45420113, 45420114, 45420116, 45420118, 45420119, 45420120, 45423315, 45423316, 45423317, 45426566, 45426567, 45426568, 45429912, 45429913, 45429914, 45429915, 45429916, 45429917, 45433196, 45433197, 45433198, 45433199, 45436520, 45436522, 45436523, 45436524, 45436526, 45436527, 45436528, 45436529, 45436530, 45436531, 45436532, 45436533, 45436534, 45439816, 45439817, 45439818, 45439819, 45439820, 45439821, 45443099, 45443100, 45443101, 45443102, 45446449, 45446450, 45446452, 45446453, 45449781, 45449782, 45449783, 45449784, 45453106, 45453107, 45453109, 45453110, 45453111, 45453113, 45453115, 45453116, 45453117, 45453118, 45456473, 45456474, 45456476, 45459820, 45459821, 45459822, 45459823, 45463259, 45463261, 45463262, 45463263, 45463264, 45463265, 45466587, 45466588, 45466591, 45466592, 45466594, 45466595, 45466596, 45466597, 45470049, 45470051, 45470052, 45473333, 45473334, 45473340, 45473341, 45473342, 45476732, 45476733, 45476735, 45480045, 45480047, 45480048, 45480049, 45483326, 45486687, 45486688, 45486690, 45486691, 45489959, 45489960, 45489961, 45489965, 45489966, 45489967, 45493232, 45493233, 45493236, 45493237, 45493240, 45493241, 45493242, 45493243, 45493244, 45496538, 45496540, 45496543, 45496544, 45496546, 45496547, 45496548, 45499858, 45499860, 45499863, 45499864, 45499867, 45499868, 45499871, 45503174, 45503177, 45503179, 45503180, 45503181, 45503182, 45503183, 45503184, 45503185, 45506456, 45506457, 45506459, 45506460, 45506461, 45509821, 45509822, 45509827, 45509828, 45513202, 45513203, 45513204, 45513205, 45513206, 45516608, 45516609, 45516612, 45516613, 45516616, 45516617, 45519839, 45519840, 45519841, 45519842, 45519843, 45523149, 45523150, 45523151, 45523153, 45523154, 45523155, 45523157, 45523158, 45533016, 45533017, 45533018, 45533019, 45533020, 45533021, 45533022, 45533023, 45537958, 45537959, 45537960, 45537961, 45537962, 45542735, 45542736, 45542737, 45542738, 45542739, 45547621, 45547622, 45547623, 45547624, 45547625, 45547626, 45547627, 45552379, 45552380, 45552381, 45552382, 45552383, 45552384, 45552385, 45552386, 45552387, 45557110, 45557111, 45557112, 45557113, 45557114, 45561947, 45561948, 45561949, 45561950, 45561951, 45561952, 45566729, 45566731, 45571654, 45571655, 45571656, 45576436, 45576437, 45576438, 45576439, 45576440, 45576441, 45576442, 45576443, 45581349, 45581350, 45581351, 45581352, 45581353, 45581354, 45581355, 45586137, 45586138, 45586139, 45586140, 45591026, 45591027, 45591028, 45591029, 45591030, 45591031, 45595793, 45595794, 45595795, 45595796, 45595797, 45595798, 45595799, 45600636, 45600637, 45600638, 45600639, 45600640, 45600641, 45600642, 45605397, 45605398, 45605400, 45605401, 45605402, 45605403, 45605404, 45605405, 45755355, 45757508, 45766051, 45766052, 45907664, 45913240, 45913241, 45913934, 45919814, 45922038, 45922039, 45922040, 45924320, 45924980, 45925324, 45925325, 45926344, 45930321, 45931224, 45931225, 45934102, 45934103, 45938368, 45938946, 45946029, 45946030, 45946031, 45947750, 45947751, 45947752, 45948564, 45951720, 45951721, 45951722, 45953138, 45954768, 45956391,
        -- existing hardcoded outcome-code exclusions (unchanged)
        4024659, 37018765, 45757124, 4326434, 4263902, 45757789, 3180817,
        439393, 314090, 433536, 1340440, 4283352, 141084, 35622939,
        4167493, 45757788
    )) AS concept_id
),
outcome_condition_codes AS (
    SELECT 'gdm'  AS outcome, explode(array(4024659, 37018765, 45757124, 4326434, 4263902, 45757789, 3180817)) AS concept_id UNION ALL
    SELECT 'pe'   AS outcome, explode(array(439393, 314090, 433536, 1340440, 4283352, 141084, 35622939)) AS concept_id UNION ALL
    SELECT 'ghtn' AS outcome, explode(array(4167493, 45757788)) AS concept_id
),
outcome_procedure_codes AS (
    SELECT 'gdm' AS outcome, explode(array(44807267)) AS concept_id
),
cohort AS (
    SELECT DISTINCT person_id, inferred_episode_start, inferred_episode_end, recorded_episode_start, recorded_episode_end
    FROM catalog.schema.step7_filtered_cohort
),

excluded_persons AS (
    SELECT DISTINCT co.person_id, c.inferred_episode_start, c.inferred_episode_end
    FROM catalog.omop.condition_occurrence co
    JOIN cohort c 
        ON co.person_id = c.person_id
    JOIN control_additional_exclusion_codes exc
        ON co.condition_source_concept_id = exc.concept_id
),
all_conditions AS (
    SELECT 
        oc.outcome,
        co.person_id,
        c.inferred_episode_start,
        c.recorded_episode_start,
        c.inferred_episode_end,
        c.recorded_episode_end,
        MIN(co.condition_start_date)     AS dx_date,
        MIN(co.condition_start_datetime) AS dx_datetime
    FROM catalog.omop.condition_occurrence co
    JOIN cohort c 
        ON co.person_id = c.person_id
        AND co.condition_start_date BETWEEN LEAST(c.inferred_episode_start, c.recorded_episode_start) AND GREATEST(c.inferred_episode_end, c.recorded_episode_end) + INTERVAL 6 weeks
    JOIN outcome_condition_codes oc 
        ON co.condition_concept_id = oc.concept_id
        OR co.condition_source_concept_id = oc.concept_id
    GROUP BY oc.outcome, co.person_id, c.inferred_episode_start, c.inferred_episode_end, c.recorded_episode_start, c.recorded_episode_end
),
all_procedures AS (
    SELECT 
        op.outcome,
        po.person_id,
        c.inferred_episode_start,
        c.inferred_episode_end,
        c.recorded_episode_start,
        c.recorded_episode_end,
        MIN(po.procedure_date)     AS dx_date,
        MIN(po.procedure_datetime) AS dx_datetime
    FROM catalog.omop.procedure_occurrence po
    JOIN cohort c 
        ON po.person_id = c.person_id
        AND po.procedure_date BETWEEN LEAST(c.inferred_episode_start, c.recorded_episode_start) AND GREATEST(c.inferred_episode_end, c.recorded_episode_end) + INTERVAL 6 weeks
    JOIN outcome_procedure_codes op 
        ON po.procedure_concept_id = op.concept_id
        OR po.procedure_source_concept_id = op.concept_id
    GROUP BY op.outcome, po.person_id, c.inferred_episode_start, c.inferred_episode_end, c.recorded_episode_end, c.recorded_episode_start
),
all_outcomes AS (
    SELECT outcome, person_id, inferred_episode_start, inferred_episode_end,
        MIN(dx_date)     AS dx_date,
        MIN(dx_datetime) AS dx_datetime
    FROM (
        SELECT * FROM all_conditions
        UNION ALL
        SELECT * FROM all_procedures
    )
    GROUP BY outcome, person_id, inferred_episode_start, inferred_episode_end
),

severe_htn AS (
    SELECT
        m.person_id,
        c.inferred_episode_start, c.inferred_episode_end,
        MIN(m.measurement_date) AS severe_htn_date
    FROM catalog.omop.measurement m
    JOIN cohort c
        ON m.person_id = c.person_id
        AND m.measurement_date >= LEAST(c.inferred_episode_start, c.recorded_episode_start) + INTERVAL 140 days
        AND m.measurement_date <= GREATEST(c.inferred_episode_end, c.recorded_episode_end)
    WHERE (m.measurement_concept_id = 3004249 AND m.value_as_number >= 160)
       OR (m.measurement_concept_id = 3012888 AND m.value_as_number >= 110)
    GROUP BY m.person_id, c.inferred_episode_start, c.inferred_episode_end
),
qualifying_htn_readings AS (
    SELECT
        m.person_id,
        c.inferred_episode_start, c.inferred_episode_end,
        m.measurement_datetime
    FROM catalog.omop.measurement m
    JOIN cohort c
        ON m.person_id = c.person_id
        AND m.measurement_date >= LEAST(c.inferred_episode_start, c.recorded_episode_start) + INTERVAL 140 days
        AND m.measurement_date <= GREATEST(c.inferred_episode_end, c.recorded_episode_end)
    WHERE (m.measurement_concept_id = 3004249 AND m.value_as_number >= 140)
       OR (m.measurement_concept_id = 3012888 AND m.value_as_number >= 90)
),
two_reading_htn AS (
    SELECT
        a.person_id, a.inferred_episode_start, a.inferred_episode_end,
        MIN(b.measurement_datetime) AS nonsevere_htn_date
    FROM qualifying_htn_readings a
    JOIN qualifying_htn_readings b
        ON a.person_id = b.person_id
        AND a.inferred_episode_start = b.inferred_episode_start
        AND a.inferred_episode_end = b.inferred_episode_end
        AND b.measurement_datetime >= a.measurement_datetime + INTERVAL 4 hours
    GROUP BY a.person_id, a.inferred_episode_start, a.inferred_episode_end
),
htn_final AS (
    SELECT
        c.person_id, c.inferred_episode_start, c.inferred_episode_end,
        LEAST(
            COALESCE(sh.severe_htn_date, '2099-01-01'),
            COALESCE(tr.nonsevere_htn_date, '2099-01-01')
        ) AS htn_dx_date
    FROM cohort c
    LEFT JOIN severe_htn sh
        ON c.person_id = sh.person_id
        AND c.inferred_episode_start = sh.inferred_episode_start
        AND c.inferred_episode_end = sh.inferred_episode_end
    LEFT JOIN two_reading_htn tr
        ON c.person_id = tr.person_id
        AND c.inferred_episode_start = tr.inferred_episode_start
        AND c.inferred_episode_end = tr.inferred_episode_end
    WHERE sh.person_id IS NOT NULL OR tr.person_id IS NOT NULL
),
proteinuria_24h AS (
    SELECT
        m.person_id,
        c.inferred_episode_start, c.inferred_episode_end,
        MIN(m.measurement_date) AS proteinuria_date
    FROM catalog.omop.measurement m
    JOIN cohort c
        ON m.person_id = c.person_id
        AND m.measurement_date >= LEAST(c.inferred_episode_start, c.recorded_episode_start) + INTERVAL 140 days
        AND m.measurement_date <= GREATEST(c.inferred_episode_end, c.recorded_episode_end)
    WHERE m.measurement_concept_id IN (3014051, 3035511, 40760845, 3037121, 3011705, 3020876)
      AND m.value_as_number >= 300
    GROUP BY m.person_id, c.inferred_episode_start, c.inferred_episode_end
),
protein_creatinine AS (
    SELECT
        m.person_id,
        c.inferred_episode_start, c.inferred_episode_end,
        MIN(m.measurement_date) AS proteinuria_date
    FROM catalog.omop.measurement m
    JOIN cohort c
        ON m.person_id = c.person_id
        AND m.measurement_date >= LEAST(c.inferred_episode_start, c.recorded_episode_start) + INTERVAL 140 days
        AND m.measurement_date <= GREATEST(c.inferred_episode_end, c.recorded_episode_end)
    WHERE m.measurement_concept_id IN (3001582, 3037791)
      AND m.value_as_number >= 0.3
    GROUP BY m.person_id, c.inferred_episode_start, c.inferred_episode_end
),
proteinuria_final AS (
    SELECT
        c.person_id, c.inferred_episode_start, c.inferred_episode_end,
        LEAST(
            COALESCE(p24.proteinuria_date, '2099-01-01'),
            COALESCE(pcr.proteinuria_date, '2099-01-01')
        ) AS proteinuria_dx_date
    FROM cohort c
    LEFT JOIN proteinuria_24h p24
        ON c.person_id = p24.person_id
        AND c.inferred_episode_start = p24.inferred_episode_start
        AND c.inferred_episode_end = p24.inferred_episode_end
    LEFT JOIN protein_creatinine pcr
        ON c.person_id = pcr.person_id
        AND c.inferred_episode_start = pcr.inferred_episode_start
        AND c.inferred_episode_end = pcr.inferred_episode_end
    WHERE p24.person_id IS NOT NULL OR pcr.person_id IS NOT NULL
),
inferred_pe_final AS (
    SELECT
        h.person_id, h.inferred_episode_start, h.inferred_episode_end,
        GREATEST(h.htn_dx_date, pr.proteinuria_dx_date) AS inferred_pe_dx_date
    FROM htn_final h
    JOIN proteinuria_final pr
        ON h.person_id = pr.person_id
        AND h.inferred_episode_start = pr.inferred_episode_start
        AND h.inferred_episode_end = pr.inferred_episode_end
)

SELECT
    pe.*,
    CASE WHEN ghtn.person_id IS NOT NULL THEN 1 ELSE 0 END AS has_ghtn,
    ghtn.dx_date     AS ghtn_dx_date,
    ghtn.dx_datetime AS ghtn_dx_datetime,
    CASE WHEN gdm.person_id IS NOT NULL THEN 1 ELSE 0 END AS has_gdm,
    gdm.dx_date      AS gdm_dx_date,
    gdm.dx_datetime  AS gdm_dx_datetime,
    CASE WHEN pe_apo.person_id IS NOT NULL THEN 1 ELSE 0 END AS has_pe,
    pe_apo.dx_date      AS pe_dx_date,
    pe_apo.dx_datetime  AS pe_dx_datetime,
    CASE WHEN inferred_pe.person_id IS NOT NULL THEN 1 ELSE 0 END AS has_inferred_pe,
    inferred_pe.inferred_pe_dx_date AS inferred_pe_dx_date,

    CASE 
        WHEN ghtn.person_id IS NULL AND gdm.person_id IS NULL AND pe_apo.person_id IS NULL 
             AND inferred_pe.person_id IS NULL
        THEN NULL
        ELSE LEAST(
            COALESCE(ghtn.dx_date, '2099-01-01'),
            COALESCE(gdm.dx_date, '2099-01-01'),
            COALESCE(pe_apo.dx_date, '2099-01-01'),
            COALESCE(inferred_pe.inferred_pe_dx_date, '2099-01-01')
        )
    END AS earliest_apo_dx_date,

    CASE 
        WHEN ghtn.person_id IS NULL 
         AND gdm.person_id IS NULL 
         AND pe_apo.person_id IS NULL
         AND inferred_pe.person_id IS NULL
         AND excl.person_id IS NULL
        THEN 1 
        ELSE 0 
    END AS is_control

FROM catalog.schema.step7_filtered_cohort pe
LEFT JOIN all_outcomes ghtn
    ON pe.person_id = ghtn.person_id
    AND pe.inferred_episode_start = ghtn.inferred_episode_start
    AND pe.inferred_episode_end = ghtn.inferred_episode_end
    AND ghtn.outcome = 'ghtn'
LEFT JOIN all_outcomes gdm
    ON pe.person_id = gdm.person_id
    AND pe.inferred_episode_start = gdm.inferred_episode_start
    AND pe.inferred_episode_end = gdm.inferred_episode_end
    AND gdm.outcome = 'gdm'
LEFT JOIN all_outcomes pe_apo
    ON pe.person_id = pe_apo.person_id
    AND pe.inferred_episode_start = pe_apo.inferred_episode_start
    AND pe.inferred_episode_end = pe_apo.inferred_episode_end
    AND pe_apo.outcome = 'pe'
LEFT JOIN inferred_pe_final inferred_pe
    ON pe.person_id = inferred_pe.person_id
    AND pe.inferred_episode_start = inferred_pe.inferred_episode_start
    AND pe.inferred_episode_end = inferred_pe.inferred_episode_end
LEFT JOIN excluded_persons excl
    ON pe.person_id = excl.person_id
    AND pe.inferred_episode_start = excl.inferred_episode_start
    AND pe.inferred_episode_end = excl.inferred_episode_end

In [0]:
%sql
SELECT 
    has_ghtn,
    has_gdm,
    has_pe,
    is_control,
    has_inferred_pe,
    COUNT(*) AS n
FROM catalog.schema.step7_filtered_cohort_with_apos
-- GROUP BY has_ghtn, has_gdm, has_pe, is_control
-- ORDER BY has_ghtn, has_gdm, has_pe, is_control;
GROUP BY has_ghtn, has_gdm, has_pe, is_control, has_inferred_pe
ORDER BY has_ghtn, has_gdm, has_pe, is_control, has_inferred_pe;

In [0]:
%sql
-- drop people from step7_filtered_cohort_with_apos who are neither a case nor a control
DELETE FROM catalog.schema.step7_filtered_cohort_with_apos
-- WHERE is_control = 0 AND has_ghtn = 0 AND has_gdm = 0 AND has_pe = 0;
WHERE is_control = 0 AND has_ghtn = 0 AND has_gdm = 0 AND has_pe = 0 AND has_inferred_pe = 0;

In [0]:
%sql
-- -- partition condition_occurrence for n3c cohort  condition
CREATE OR REPLACE TABLE catalog.schema.condition_occurrence_cohort
CLUSTER BY AUTO
AS
SELECT co.*
FROM catalog.omop.condition_occurrence co
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON co.person_id = pe.person_id
    AND co.condition_start_date >= DATE_SUB(pe.inferred_episode_start, 180)
    AND CASE
        WHEN pe.is_control = 1 THEN co.condition_start_date < pe.inferred_episode_end
        ELSE co.condition_start_date < pe.earliest_apo_dx_date
    END;


-- partition procedure_occurrence for n3c cohort - procedure
CREATE OR REPLACE TABLE catalog.schema.procedure_occurrence_cohort
CLUSTER BY AUTO
AS
SELECT po.*
FROM catalog.omop.procedure_occurrence po
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON po.person_id = pe.person_id
    AND po.procedure_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN po.procedure_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE po.procedure_date < pe.earliest_apo_dx_date
    END;

-- person (no date column, just person-level filter)
CREATE OR REPLACE TABLE catalog.schema.person_cohort
CLUSTER BY AUTO
AS
SELECT p.*
FROM catalog.omop.person p
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON p.person_id = pe.person_id;

-- drug_exposure
CREATE OR REPLACE TABLE catalog.schema.drug_exposure_cohort
CLUSTER BY AUTO
AS
SELECT de.*
FROM catalog.omop.drug_exposure de
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON de.person_id = pe.person_id
    AND de.drug_exposure_start_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN de.drug_exposure_start_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE de.drug_exposure_start_date < pe.earliest_apo_dx_date
    END;

-- visit_detail
CREATE OR REPLACE TABLE catalog.schema.visit_detail_cohort
CLUSTER BY AUTO
AS
SELECT d.*
FROM catalog.omop.visit_detail d
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON d.person_id = pe.person_id
    AND d.visit_detail_start_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN d.visit_detail_start_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE d.visit_detail_start_date < pe.earliest_apo_dx_date
    END;

-- visit_occurrence
CREATE OR REPLACE TABLE catalog.schema.visit_occurrence_cohort
CLUSTER BY AUTO
AS
SELECT vo.*
FROM catalog.omop.visit_occurrence vo
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON vo.person_id = pe.person_id
    AND vo.visit_start_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN vo.visit_start_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE vo.visit_start_date < pe.earliest_apo_dx_date
    END;

-- death (death_date)
CREATE OR REPLACE TABLE catalog.schema.death_cohort
CLUSTER BY AUTO
AS
SELECT d.*
FROM catalog.omop.death d
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON d.person_id = pe.person_id
    AND d.death_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN d.death_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE d.death_date < pe.earliest_apo_dx_date
    END;

-- device_exposure
CREATE OR REPLACE TABLE catalog.schema.device_exposure_cohort
CLUSTER BY AUTO
AS
SELECT de.*
FROM catalog.omop.device_exposure de
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON de.person_id = pe.person_id
    AND de.device_exposure_start_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN de.device_exposure_start_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE de.device_exposure_start_date < pe.earliest_apo_dx_date
    END;

-- measurement
CREATE OR REPLACE TABLE catalog.schema.measurement_cohort
CLUSTER BY AUTO
AS
SELECT m.*
FROM catalog.omop.measurement m
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON m.person_id = pe.person_id
    AND m.measurement_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN m.measurement_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE m.measurement_date < pe.earliest_apo_dx_date
    END;

-- observation
CREATE OR REPLACE TABLE catalog.schema.observation_cohort
CLUSTER BY AUTO
AS
SELECT o.*
FROM catalog.omop.observation o
JOIN catalog.schema.step7_filtered_cohort_with_apos pe
    ON o.person_id = pe.person_id
    AND o.observation_date >= DATE_SUB(LEAST(pe.inferred_episode_start, pe.recorded_episode_start), 180)
    AND CASE
        WHEN pe.is_control = 1 THEN o.observation_date < GREATEST(pe.inferred_episode_end, pe.recorded_episode_end)
        ELSE o.observation_date < pe.earliest_apo_dx_date
    END;



In [0]:
%sql
-- drop rows from measurement_cohort where measurement_concept_id = 0 OR unit_concept_id = 0
DELETE FROM catalog.schema.measurement_cohort
-- WHERE measurement_concept_id = 0 OR unit_concept_id = 0;
WHERE measurement_concept_id = 0;

-- SELECT COUNT(*) FROM catalog.schema.measurement_cohort
-- WHERE unit_concept_id = 0;

-- 16,558,585 total lab measurements
-- 849,303 have no known measurement concept id (0)
-- 3,469,867 have no known unit (0)

-- previewing which labs will be dropped for low-prevalence reasons
-- SELECT 
--     m.measurement_concept_id,
--     c.concept_name,
--     COUNT(DISTINCT m.person_id)       AS n_patients,
--     COUNT(DISTINCT m.unit_concept_id) AS n_units
-- FROM catalog.schema.measurement_cohort m
-- JOIN catalog.omop.concept c
--     ON c.concept_id = m.measurement_concept_id
-- WHERE m.measurement_concept_id != 0
-- GROUP BY m.measurement_concept_id, c.concept_name
-- HAVING COUNT(DISTINCT m.person_id) < (
--     SELECT COUNT(DISTINCT person_id) * 0.2 
--     FROM catalog.schema.step7_filtered_cohort_with_apos
-- )
-- AND COUNT(DISTINCT m.unit_concept_id) > 1
-- AND NOT (
--     COUNT(DISTINCT m.unit_concept_id) = 2
--     AND SUM(CASE WHEN m.unit_concept_id = 0 THEN 1 ELSE 0 END) > 0
-- )
-- ORDER BY n_patients ASC;

-- SELECT COUNT(DISTINCT measurement_concept_id)
-- FROM catalog.schema.measurement_cohort

-- SELECT COUNT(*) AS n_low_prevalence_and_unharmonized_concepts
-- FROM (
--     SELECT 
--         m.measurement_concept_id,
--         COUNT(DISTINCT m.person_id)       AS n_patients,
--         COUNT(DISTINCT m.unit_concept_id) AS n_units
--     FROM catalog.schema.measurement_cohort m
--     WHERE m.measurement_concept_id != 0
--     GROUP BY m.measurement_concept_id
--     HAVING COUNT(DISTINCT m.person_id) < (
--         SELECT COUNT(DISTINCT person_id) * 0.2 
--         FROM catalog.schema.step7_filtered_cohort_with_apos
--     )
--     AND COUNT(DISTINCT m.unit_concept_id) > 1
--     AND NOT (
--         COUNT(DISTINCT m.unit_concept_id) = 2
--         AND SUM(CASE WHEN m.unit_concept_id = 0 THEN 1 ELSE 0 END) > 0
--     )
-- ) low_prev_unharmonized

-- DELETE FROM catalog.schema.measurement_cohort
-- WHERE measurement_concept_id IN (
--     SELECT measurement_concept_id
--     FROM (
--         SELECT 
--             m.measurement_concept_id,
--             COUNT(DISTINCT m.person_id)       AS n_patients,
--             COUNT(DISTINCT m.unit_concept_id) AS n_units
--         FROM catalog.schema.measurement_cohort m
--         WHERE m.measurement_concept_id != 0
--         GROUP BY m.measurement_concept_id
--         HAVING COUNT(DISTINCT m.person_id) < (
--             SELECT COUNT(DISTINCT person_id) * 0.2 
--             FROM catalog.schema.step7_filtered_cohort_with_apos
--         )
--         AND COUNT(DISTINCT m.unit_concept_id) > 1
--         AND NOT (
--             COUNT(DISTINCT m.unit_concept_id) = 2
--             AND SUM(CASE WHEN m.unit_concept_id = 0 THEN 1 ELSE 0 END) > 0
--         )
--     ) low_prev_unharmonized
-- )

In [0]:
%sql
DELETE FROM catalog.schema.measurement_cohort
WHERE (measurement_concept_id = 3035583
    AND unit_concept_id = 8786)
OR (measurement_concept_id = 3000905
    AND unit_concept_id = 8786)
OR (measurement_concept_id = 3020416
    AND unit_concept_id IN (9433, 8549, 8786))
OR (measurement_concept_id = 3024929
    AND unit_concept_id = 9433)
OR (measurement_concept_id = 3020891
    AND unit_concept_id = 9471)
OR (measurement_concept_id = 3011397 
    AND unit_concept_id = 8786);

UPDATE catalog.schema.measurement_cohort
SET 
    value_as_number = value_as_number / 1000.0,
    unit_concept_id = 8713,
    value_source_value = CAST(value_as_number / 1000.0 AS STRING)
WHERE measurement_concept_id = 3024561
  AND unit_concept_id = 8840;

UPDATE catalog.schema.measurement_cohort
SET 
    value_as_number = value_as_number / 1000.0,
    unit_concept_id = 8848,
    value_source_value = CAST(value_as_number / 1000.0 AS STRING)
WHERE measurement_concept_id IN (3033575, 3004327, 3013429, 3028615, 3013650) 
  AND unit_concept_id = 8784;

UPDATE catalog.schema.measurement_cohort
SET 
    value_as_number = (value_as_number * 9/5) + 32,
    unit_concept_id = 9289,
    value_source_value = CAST((value_as_number * 9/5) + 32 AS STRING)
WHERE measurement_concept_id = 3020891
AND unit_concept_id = 586323;

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# ---------------------------------------------------------------
# Table-specific config: (table_name, concept_id_column)
# ---------------------------------------------------------------
tables_config = [
    {"table_name": "condition_occurrence", "concept_id_col": "condition_concept_id"},
    {"table_name": "procedure_occurrence",  "concept_id_col": "procedure_concept_id"},
    {"table_name": "drug_exposure",         "concept_id_col": "drug_concept_id"},
    {"table_name": "measurement",           "concept_id_col": "measurement_concept_id"},
    {"table_name": "device_exposure",       "concept_id_col": "device_concept_id"},
]

CONCEPT_SCHEMA = "catalog.omop"   # concept / concept_ancestor still live here
SOURCE_SCHEMA = "catalog.schema"  # the already-partitioned _cohort tables
DEST_SCHEMA = "catalog.schema"
SPARSITY_THRESHOLD = 0.20

def build_rollup_query(table_name, concept_id_col):
    return f"""
    WITH cohort_size AS (
        SELECT COUNT(DISTINCT person_id) AS n_total FROM catalog.schema.step7_filtered_cohort_with_apos
    ),
    code_prevalence AS (
        SELECT 
            m.{concept_id_col} AS concept_id,
            COUNT(DISTINCT m.person_id) AS n_patients,
            c.domain_id
        FROM {SOURCE_SCHEMA}.{table_name}_cohort m
        JOIN {CONCEPT_SCHEMA}.concept c ON c.concept_id = m.{concept_id_col}
        GROUP BY m.{concept_id_col}, c.domain_id
    ),
    all_codes AS (
        SELECT 
            cp.concept_id, 
            cp.domain_id, 
            cp.n_patients,
            CASE WHEN cp.n_patients <= {SPARSITY_THRESHOLD} * cs.n_total THEN true ELSE false END AS is_sparse
        FROM code_prevalence cp
        CROSS JOIN cohort_size cs
    ),
    self_mapped AS (
        SELECT 
            ac.concept_id,
            CASE WHEN tk.code IS NOT NULL THEN true ELSE false END AS code_itself_mapped
        FROM all_codes ac
        LEFT JOIN {CONCEPT_SCHEMA}.concept c ON c.concept_id = ac.concept_id
        LEFT JOIN catalog.schema.clmbr_tokenizer_known_codes tk 
            ON tk.code = CONCAT(c.vocabulary_id, '/', c.concept_code)
    ),
    codes_needing_rollup AS (
        SELECT ac.concept_id, ac.domain_id, ac.n_patients, ac.is_sparse
        FROM all_codes ac
        JOIN self_mapped sm ON sm.concept_id = ac.concept_id
        WHERE sm.code_itself_mapped = false
    ),
    ancestor_narrowness AS (
        SELECT 
            ancestor_concept_id,
            COUNT(DISTINCT descendant_concept_id) AS n_total_descendants
        FROM {CONCEPT_SCHEMA}.concept_ancestor
        GROUP BY ancestor_concept_id
    ),
    level1_ancestors AS (
        SELECT
            cnr.concept_id AS target_concept_id,
            ca.ancestor_concept_id,
            c.concept_name AS ancestor_name,
            an.n_total_descendants,
            ROW_NUMBER() OVER (
                PARTITION BY cnr.concept_id 
                ORDER BY an.n_total_descendants ASC
            ) AS narrowness_rank
        FROM codes_needing_rollup cnr
        JOIN {CONCEPT_SCHEMA}.concept_ancestor ca 
            ON ca.descendant_concept_id = cnr.concept_id AND ca.min_levels_of_separation = 1
        JOIN {CONCEPT_SCHEMA}.concept c ON c.concept_id = ca.ancestor_concept_id
        JOIN catalog.schema.clmbr_tokenizer_known_codes tk ON tk.code = CONCAT(c.vocabulary_id, '/', c.concept_code)
        JOIN ancestor_narrowness an ON an.ancestor_concept_id = ca.ancestor_concept_id
        WHERE c.domain_id = cnr.domain_id
    ),
    level2_ancestors AS (
        SELECT
            cnr.concept_id AS target_concept_id,
            ca.ancestor_concept_id,
            c.concept_name AS ancestor_name,
            an.n_total_descendants,
            ROW_NUMBER() OVER (
                PARTITION BY cnr.concept_id 
                ORDER BY an.n_total_descendants ASC
            ) AS narrowness_rank
        FROM codes_needing_rollup cnr
        JOIN {CONCEPT_SCHEMA}.concept_ancestor ca 
            ON ca.descendant_concept_id = cnr.concept_id AND ca.min_levels_of_separation = 2
        JOIN {CONCEPT_SCHEMA}.concept c ON c.concept_id = ca.ancestor_concept_id
        JOIN catalog.schema.clmbr_tokenizer_known_codes tk ON tk.code = CONCAT(c.vocabulary_id, '/', c.concept_code)
        JOIN ancestor_narrowness an ON an.ancestor_concept_id = ca.ancestor_concept_id
        WHERE c.domain_id = cnr.domain_id
          AND cnr.concept_id NOT IN (SELECT target_concept_id FROM level1_ancestors)
    ),
    all_ancestor_candidates AS (
        SELECT target_concept_id, ancestor_concept_id, ancestor_name, n_total_descendants, narrowness_rank, 1 AS level
        FROM level1_ancestors
        UNION ALL
        SELECT target_concept_id, ancestor_concept_id, ancestor_name, n_total_descendants, narrowness_rank, 2 AS level
        FROM level2_ancestors
    )
    SELECT 
        ac.concept_id AS original_concept_id,
        orig_c.concept_name AS original_concept_name,
        ac.domain_id,
        ac.n_patients,
        ac.is_sparse,
        sm.code_itself_mapped,
        aac.level AS rollup_level,
        aac.ancestor_concept_id AS candidate_ancestor_id,
        aac.ancestor_name AS candidate_ancestor_name,
        aac.n_total_descendants AS candidate_ancestor_narrowness,
        aac.narrowness_rank,
        CASE 
            WHEN sm.code_itself_mapped = true THEN 'mapped_keep_as_is'
            WHEN sm.code_itself_mapped = false AND aac.ancestor_concept_id IS NOT NULL THEN 'unmapped_rolled_up'
            WHEN sm.code_itself_mapped = false AND aac.ancestor_concept_id IS NULL THEN 'unmapped_no_rollup_flag_for_drop'
        END AS resolution
    FROM all_codes ac
    JOIN {CONCEPT_SCHEMA}.concept orig_c ON orig_c.concept_id = ac.concept_id
    JOIN self_mapped sm ON sm.concept_id = ac.concept_id
    LEFT JOIN all_ancestor_candidates aac ON aac.target_concept_id = ac.concept_id
    ORDER BY ac.n_patients ASC, ac.concept_id, aac.narrowness_rank
    """


rollup_summaries = {}

for cfg in tables_config:
    table_name = cfg["table_name"]
    concept_id_col = cfg["concept_id_col"]

    print(f"\n{'='*80}\nProcessing table: {table_name}\n{'='*80}")

    query = build_rollup_query(table_name, concept_id_col)
    rollup_df = spark.sql(query).toPandas()

    rolled_up_df = rollup_df[
        (rollup_df['resolution'] == 'unmapped_rolled_up') & (rollup_df['narrowness_rank'] == 1)
    ].copy()

    status_df = rollup_df.drop_duplicates(subset=['original_concept_id'], keep='first')[
        ['original_concept_id', 'resolution']
    ]

    n_total_concepts = status_df['original_concept_id'].nunique()
    n_mapped_keep = (status_df['resolution'] == 'mapped_keep_as_is').sum()
    n_rolled_up = (status_df['resolution'] == 'unmapped_rolled_up').sum()
    n_flag_drop = (status_df['resolution'] == 'unmapped_no_rollup_flag_for_drop').sum()

    print(f"Total distinct concept_ids: {n_total_concepts}")
    print(f"  mapped_keep_as_is: {n_mapped_keep}")
    print(f"  unmapped_rolled_up: {n_rolled_up}")
    print(f"  unmapped_no_rollup_flag_for_drop: {n_flag_drop}")

    if len(rolled_up_df) > 0:
        print(f"\nFull list of rolled-up concept mappings for {table_name}:")
        print(rolled_up_df[[
            'original_concept_id', 'original_concept_name', 
            'candidate_ancestor_id', 'candidate_ancestor_name'
        ]].to_string())

    rollup_summaries[table_name] = {
        'full_df': rollup_df,
        'rolled_up_df': rolled_up_df,
        'status_df': status_df
    }

    remap_dict = dict(zip(rolled_up_df['original_concept_id'], rolled_up_df['candidate_ancestor_id']))

    # ---------------------------------------------------------------
    # source IS the destination now (catalog.schema.{table_name}_cohort),
    # so we still need the temp-table pattern to avoid reading/writing 
    # the same table location simultaneously
    # ---------------------------------------------------------------
    source_df = spark.table(f"{SOURCE_SCHEMA}.{table_name}_cohort")

    if remap_dict:
        remap_pd = pd.DataFrame(
            [{'orig_id': k, 'new_id': v} for k, v in remap_dict.items()]
        )
        remap_spark = spark.createDataFrame(remap_pd)

        updated_df = (
            source_df.alias("src")
            .join(remap_spark.alias("rm"), source_df[concept_id_col] == remap_spark["orig_id"], how="left")
            .withColumn(
                concept_id_col,
                F.coalesce(F.col("rm.new_id"), F.col(f"src.{concept_id_col}"))
            )
            .drop("orig_id", "new_id")
        )
    else:
        updated_df = source_df

    dest_table = f"{DEST_SCHEMA}.{table_name}_cohort"
    temp_table = f"{DEST_SCHEMA}.__tmp_{table_name}_cohort"

    updated_df.write.mode("overwrite").saveAsTable(temp_table)
    spark.sql(f"CREATE OR REPLACE TABLE {dest_table} AS SELECT * FROM {temp_table}")
    spark.sql(f"DROP TABLE IF EXISTS {temp_table}")

    print(f"\nSaved updated table to: {dest_table}")

print("\n\nAll tables processed.")

In [0]:
# save partitioned tables as csvs for etl
OUTPUT_DIR = "/path/to/exports/omop_export"

def export_table_to_csv(table_name, source_table, column_renames=None):
    print(f"Exporting {table_name}...")
    df = spark.table(source_table)
    if column_renames:
        for old_name, new_name in column_renames.items():
            df = df.withColumnRenamed(old_name, new_name)
    df.write.mode("overwrite") \
        .option("header", "true") \
        .option("quoteAll", "false") \
        .option("escape", '"') \
        .option("quote", '"') \
        .csv(f"{OUTPUT_DIR}/{table_name}")
    print(f"Done: {table_name}")

export_table_to_csv("person", "catalog.schema.person_cohort")
export_table_to_csv("condition_occurrence", "catalog.schema.condition_occurrence_cohort")
export_table_to_csv("procedure_occurrence", "catalog.schema.procedure_occurrence_cohort")
export_table_to_csv("drug_exposure",        "catalog.schema.drug_exposure_cohort")
export_table_to_csv("visit_occurrence",     "catalog.schema.visit_occurrence_cohort")
export_table_to_csv("visit_detail",         "catalog.schema.visit_detail_cohort")
export_table_to_csv("measurement",          "catalog.schema.measurement_cohort")
export_table_to_csv("observation",          "catalog.schema.observation_cohort")
export_table_to_csv("device_exposure",      "catalog.schema.device_exposure_cohort")
export_table_to_csv("death",                "catalog.schema.death_cohort")
export_table_to_csv(
    "visit_occurrence", 
    "catalog.schema.visit_occurrence_cohort",
    column_renames={"discharge_to_concept_id": "discharged_to_concept_id"}
)
export_table_to_csv("concept",                "catalog.omop.concept")
export_table_to_csv("concept_relationship", "catalog.omop.concept_relationship")

print("All tables exported!")
